# 07 Anomalies (BQ7): R version

**BQ7:** Which days and which transactions are unusual?

Same data, rules and features as `07_anomalies_py.ipynb` (see it for the full explanation). The daily residuals,
z-scores, control limits and IQR fences must match exactly. Isolation Forest uses R's `isotree` (standard,
axis-parallel splits: `ndim = 1`), which is randomised and implemented differently from scikit-learn, so the scores
are compared by rank correlation and by the overlap of the flagged invoices.

## Setup and data

In [ ]:
suppressPackageStartupMessages({
  library(arrow); library(dplyr); library(tidyr); library(ggplot2); library(scales); library(isotree)
})
options(dplyr.summarise.inform = FALSE, scipen = 999, digits = 4, width = 160,
        repr.plot.width = 11, repr.plot.height = 4)
Sys.setenv(TZ = "UTC")

root <- if (dir.exists("data")) "." else ".."
proc <- file.path(root, "data", "processed")
fig  <- file.path(root, "reports", "figures")
dir.create(fig, showWarnings = FALSE, recursive = TRUE)

START <- as.POSIXct("2009-12-01", tz = "UTC")
END   <- as.POSIXct("2011-12-01", tz = "UTC")
Z_LIMIT <- 3; IQR_K <- 3; TOP_SHARE <- 0.01; SEED <- 42

read_tbl <- function(name) {
  read_parquet(file.path(proc, paste0(name, ".parquet"))) |>
    mutate(date = as.Date(date)) |>
    filter(invoice_date >= START, invoice_date < END)
}
sales   <- read_tbl("sales_clean")
cancels <- read_tbl("cancellations")
cat(sprintf("Sales lines: %s   cancellation lines: %s\n", comma(nrow(sales)), comma(nrow(cancels))))
print(table(sales$line_type))

# Part A: Unusual days (`log(net revenue) ~ weekday + year-month`; z-score, I-MR chart and IQR rules)

In [ ]:
prod <- sales |> filter(line_type == "product")
canc <- cancels |> filter(line_type == "product")
trading <- unique(prod$date)
daily <- bind_rows(prod |> select(date, revenue), canc |> select(date, revenue)) |>
  group_by(date) |> summarise(net = sum(revenue)) |>
  filter(date %in% trading)
n_nonpos <- sum(daily$net <= 0)
daily <- daily |> filter(net > 0) |> arrange(date) |>
  mutate(weekday = format(date, "%A"), year_month = format(date, "%Y-%m"))

fit <- lm(log(net) ~ factor(weekday) + factor(year_month), data = daily)
daily <- daily |>
  mutate(expected = exp(fitted(fit)), resid = residuals(fit), z = resid / sd(resid))

mr_bar <- mean(abs(diff(daily$resid)))
cl <- mean(daily$resid); ucl <- cl + 2.66 * mr_bar; lcl <- cl - 2.66 * mr_bar
q <- quantile(daily$resid, c(0.25, 0.75))
lo_f <- q[[1]] - IQR_K * diff(q); hi_f <- q[[2]] + IQR_K * diff(q)

daily <- daily |>
  mutate(flag_z = abs(z) > Z_LIMIT,
         flag_chart = resid > ucl | resid < lcl,
         flag_iqr = resid > hi_f | resid < lo_f,
         n_rules = flag_z + flag_chart + flag_iqr)

cat(sprintf("Trading days: %s (excluded %d with net revenue <= 0)   R² of the daily model: %.3f\n",
            comma(nrow(daily)), n_nonpos, summary(fit)$r.squared))
cat(sprintf("Residual SD: %.4f   control limits: [%.4f, %.4f]   IQR fences: [%.4f, %.4f]\n",
            sd(daily$resid), lcl, ucl, lo_f, hi_f))
cat(sprintf("Days flagged: z-score %d, control chart %d, IQR %d, by all three %d, by at least one %d\n\n",
            sum(daily$flag_z), sum(daily$flag_chart), sum(daily$flag_iqr), sum(daily$n_rules == 3), sum(daily$n_rules >= 1)))
cat("Unusual days (largest deviation first):\n")
print(as.data.frame(daily |> filter(n_rules >= 1) |> arrange(desc(abs(z))) |>
        select(date, weekday, net, expected, resid, z, flag_z, flag_chart, flag_iqr) |> head(20)), digits = 4)

In [ ]:
flagged <- daily |> filter(n_rules >= 1)
p1 <- ggplot(daily, aes(date)) +
  geom_line(aes(y = net / 1000, colour = "Net revenue"), linewidth = 0.3) +
  geom_line(aes(y = expected / 1000, colour = "Expected (weekday + month)"), linewidth = 0.5) +
  geom_point(data = flagged, aes(y = net / 1000, colour = "Flagged"), size = 2) +
  scale_colour_manual(values = c("Net revenue" = "grey55", "Expected (weekday + month)" = "#64B5F6", "Flagged" = "#FF8A65")) +
  labs(x = NULL, y = "£k per day", colour = NULL, title = "Daily net revenue and unusual days") +
  theme_minimal(base_size = 11) + theme(legend.position = "bottom")
p2 <- ggplot(daily, aes(date, resid)) +
  geom_line(colour = "grey55", linewidth = 0.3) +
  geom_hline(yintercept = c(lcl, ucl), colour = "#FF8A65", linetype = "dashed") +
  geom_hline(yintercept = cl, colour = "#64B5F6") +
  geom_point(data = filter(daily, flag_chart), colour = "#FF8A65", size = 1.8) +
  labs(x = NULL, y = "Residual (log)", title = "Individuals control chart of residuals") +
  theme_minimal(base_size = 11)
ggsave(file.path(fig, "bq7_unusual_days_r.png"), p1, width = 11, height = 4, dpi = 150)
ggsave(file.path(fig, "bq7_control_chart_r.png"), p2, width = 11, height = 3.5, dpi = 150)
options(repr.plot.width = 11, repr.plot.height = 4); print(p1); print(p2)

# Part B: Unusual invoices (Isolation Forest on invoice features)

In [ ]:
inv <- sales |>
  mutate(nonprod_rev = if_else(line_type != "product", revenue, 0),
         cancel_rev = if_else(cancelled_exact, revenue, 0),
         price_dev = if_else(line_type == "product", abs(log(price_ratio)), NA_real_)) |>
  group_by(invoice) |>
  summarise(lines = n(), qty = sum(quantity), value = sum(revenue),
            nonprod_rev = sum(nonprod_rev), cancel_rev = sum(cancel_rev),
            max_price_dev = suppressWarnings(max(price_dev, na.rm = TRUE)),
            has_customer = as.integer(any(has_customer)), is_uk = as.integer(any(is_uk)),
            hour = min(hour), any_bulk = any(is_bulk), date = min(date)) |>
  arrange(invoice) |>
  mutate(max_price_dev = if_else(is.finite(max_price_dev), max_price_dev, 0),
         nonproduct_share = nonprod_rev / value, cancelled_share = cancel_rev / value)

X <- data.frame(log_lines = log(inv$lines), log_qty = log1p(inv$qty), log_value = log1p(inv$value),
                log_avg_price = log(inv$value / inv$qty), nonproduct_share = inv$nonproduct_share,
                max_price_dev = inv$max_price_dev, cancelled_share = inv$cancelled_share,
                has_customer = inv$has_customer, is_uk = inv$is_uk, hour = as.numeric(inv$hour))
cat(sprintf("Invoices: %s\n\n", comma(nrow(inv))))
print(round(t(sapply(X, function(v) c(mean = mean(v), sd = sd(v), min = min(v), median = median(v), max = max(v)))), 3))

In [ ]:
iso <- isolation.forest(X, ntrees = 200, sample_size = 256, ndim = 1, seed = SEED, nthreads = 1)
inv$score <- predict(iso, X)                                  # higher = more anomalous
n_top <- ceiling(TOP_SHARE * nrow(inv))
inv$flag_iforest <- rank(-inv$score, ties.method = "first") <= n_top

iqr_flag <- function(x) { q <- quantile(x, c(0.25, 0.75)); x < q[[1]] - IQR_K * diff(q) | x > q[[2]] + IQR_K * diff(q) }
inv$flag_iqr_value <- iqr_flag(X$log_value); inv$flag_iqr_qty <- iqr_flag(X$log_qty)
cat(sprintf("Isolation Forest: top %.0f%% = %d invoices flagged\n", 100 * TOP_SHARE, n_top))
cat(sprintf("IQR fences (3 x IQR): value %d, quantity %d; also flagged by Isolation Forest: %d\n\n",
            sum(inv$flag_iqr_value), sum(inv$flag_iqr_qty),
            sum(inv$flag_iforest & (inv$flag_iqr_value | inv$flag_iqr_qty))))

grp <- function(d) c(invoices = nrow(d), median_value = median(d$value), median_lines = median(d$lines),
                     has_nonproduct = mean(d$nonprod_rev > 0), has_cancelled = mean(d$cancel_rev > 0),
                     has_bulk_line = mean(d$any_bulk), has_price_outlier = mean(d$max_price_dev > log(5)),
                     no_customer = mean(d$has_customer == 0), non_uk = mean(d$is_uk == 0))
profile <- cbind(`Flagged (top 1%)` = grp(filter(inv, flag_iforest)), `All other invoices` = grp(filter(inv, !flag_iforest)))
cat("Profile of flagged vs other invoices (shares of invoices):\n"); print(round(profile, 3))

In [ ]:
cat("Ten most anomalous invoices:\n")
print(as.data.frame(inv |> arrange(desc(score)) |>
        select(invoice, date, lines, qty, value, nonproduct_share, max_price_dev, cancelled_share,
               has_customer, is_uk, hour, score) |> head(10)), digits = 3)

p3 <- ggplot(cbind(X, flag = inv$flag_iforest), aes(log_qty, log_value)) +
  geom_point(data = \(d) filter(d, !flag), colour = "grey60", size = 0.4, alpha = 0.3) +
  geom_point(data = \(d) filter(d, flag), colour = "#FF8A65", size = 1.2) +
  labs(x = "log(1 + total quantity)", y = "log(1 + invoice value)",
       title = "Invoices: size vs value, with the Isolation Forest top 1% highlighted") +
  theme_minimal(base_size = 11)
ggsave(file.path(fig, "bq7_unusual_invoices_r.png"), p3, width = 9, height = 4.5, dpi = 150)
options(repr.plot.width = 9, repr.plot.height = 4.5); p3

## Checks against Python

In [ ]:
dash <- file.path(root, "reports", "dashboard_data")
if (file.exists(file.path(dash, "anomaly_daily_residuals.csv"))) {
  py_d <- read.csv(file.path(dash, "anomaly_daily_residuals.csv")) |> mutate(date = as.Date(date))
  py_s <- read.csv(file.path(dash, "anomaly_invoice_scores.csv"), colClasses = c(invoice = "character"))
  py_days <- read.csv(file.path(dash, "anomaly_days.csv")) |> mutate(date = as.Date(date))
  d <- inner_join(daily, py_d, by = "date", suffix = c("_r", "_py"))
  cat(sprintf("Trading days: R %d, Python %d   max |residual difference|: %.1e   max |z difference|: %.1e\n",
              nrow(daily), nrow(py_d), max(abs(d$resid_r - d$resid_py)), max(abs(d$z_r - d$z_py))))
  cat(sprintf("Flagged days identical: %s\n", identical(sort(flagged$date), sort(py_days$date))))
  s <- inner_join(inv |> select(invoice, score, flag_iforest), py_s, by = "invoice", suffix = c("_r", "_py"))
  py_top <- s$invoice[rank(-s$score_py, ties.method = "first") <= n_top]
  cat(sprintf("Invoices compared: %s   Spearman correlation of Isolation Forest scores: %.3f\n",
              comma(nrow(s)), cor(s$score_r, s$score_py, method = "spearman")))
  cat(sprintf("Top 1%% flagged by both: %d of %d (%.0f%%)\n", length(intersect(py_top, s$invoice[s$flag_iforest])),
              n_top, 100 * length(intersect(py_top, s$invoice[s$flag_iforest])) / n_top))
} else {
  cat("Run 07_anomalies_py.ipynb first to create the dashboard CSVs.\n")
}